# Cloud Metrics Analytics

This notebook queries the `clean_metrics` MongoDB collection and generates:
- Time-series plots for all metrics
- Statistical summaries and correlation heatmap
- Parquet export

**Prerequisites:** Set `MONGODB_URI` in the container `.env` file.

In [ ]:
import os
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
from pymongo import MongoClient

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
sns.set_theme(style='darkgrid')

print('Libraries loaded.')

## 1. Configuration

In [ ]:
# ── Connection ──────────────────────────────────────────────────────────────
MONGODB_URI = os.environ.get('MONGODB_URI', 'mongodb://localhost:27017')
DB_NAME = 'metrics_db'
COLLECTION = 'clean_metrics'

# ── Query range ─────────────────────────────────────────────────────────────
# Change these to query a different time window
HOURS_BACK = 24            # How many hours of data to load
HOST_ID = None             # Set to a specific host_id string, or None for all hosts

# ── Export ──────────────────────────────────────────────────────────────────
EXPORTS_DIR = Path('/home/jovyan/work/exports')
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Config: last {HOURS_BACK}h | host={HOST_ID or "all"} | exports → {EXPORTS_DIR}')

## 2. Load Data from MongoDB

In [ ]:
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
db = client[DB_NAME]
collection = db[COLLECTION]

now = datetime.now(timezone.utc)
since = now - timedelta(hours=HOURS_BACK)

query = {'window_start': {'$gte': since.isoformat(), '$lte': now.isoformat()}}
if HOST_ID:
    query['host_id'] = HOST_ID

docs = list(collection.find(query, {'_id': 0}))
print(f'Loaded {len(docs)} clean_metrics documents')

if not docs:
    print('No data found. Run the Airflow DAG first, or adjust HOURS_BACK.')

In [ ]:
# Flatten nested metrics dict into a flat DataFrame
METRIC_COLS = [
    'cpu_load_percent',
    'memory_load_percent',
    'memory_used_mb',
    'network_in_bytes',
    'network_out_bytes',
    'latency_ms',
    'power_consumption_w',
]

rows = []
for doc in docs:
    row = {
        'host_id': doc['host_id'],
        'window_start': pd.to_datetime(doc['window_start'], utc=True),
        'window_end': pd.to_datetime(doc.get('window_end'), utc=True),
        'sample_count': doc.get('sample_count', 0),
    }
    for col in METRIC_COLS:
        m = doc.get('metrics', {}).get(col, {})
        row[f'{col}_mean'] = m.get('mean') if isinstance(m, dict) else m
        row[f'{col}_min'] = m.get('min') if isinstance(m, dict) else None
        row[f'{col}_max'] = m.get('max') if isinstance(m, dict) else None
    rows.append(row)

df = pd.DataFrame(rows).sort_values('window_start').reset_index(drop=True)
print(f'DataFrame shape: {df.shape}')
df.head(3)

## 3. Time-Series Plots

In [ ]:
hosts = df['host_id'].unique()
colors = plt.cm.tab10.colors

def plot_metric(ax, col, ylabel, title, formatter=None, scale=1.0):
    for i, host in enumerate(hosts):
        host_df = df[df['host_id'] == host]
        x = host_df['window_start']
        y = host_df[f'{col}_mean'] * scale
        y_min = host_df[f'{col}_min'] * scale
        y_max = host_df[f'{col}_max'] * scale
        color = colors[i % len(colors)]
        ax.plot(x, y, label=host, color=color, linewidth=1.2)
        ax.fill_between(x, y_min, y_max, alpha=0.15, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    if formatter:
        ax.yaxis.set_major_formatter(formatter)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')


fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle(f'System Metrics — Last {HOURS_BACK}h', fontsize=14, fontweight='bold')

plot_metric(axes[0, 0], 'cpu_load_percent', '% utilization', 'CPU Load')
axes[0, 0].set_ylim(0, 100)

plot_metric(axes[0, 1], 'memory_load_percent', '% utilization', 'Memory Load')
axes[0, 1].set_ylim(0, 100)

plot_metric(axes[1, 0], 'latency_ms', 'ms', 'Network Latency')

plot_metric(axes[1, 1], 'power_consumption_w', 'Watts', 'Power Consumption')

# Network dual-axis
ax_net = axes[2, 0]
ax_net2 = ax_net.twinx()
for i, host in enumerate(hosts):
    host_df = df[df['host_id'] == host]
    color = colors[i % len(colors)]
    ax_net.plot(host_df['window_start'],
                host_df['network_in_bytes_mean'] / 1024,
                label=f'{host} IN', color=color, linestyle='-')
    ax_net2.plot(host_df['window_start'],
                 host_df['network_out_bytes_mean'] / 1024,
                 label=f'{host} OUT', color=color, linestyle='--')
ax_net.set_title('Network Traffic')
ax_net.set_ylabel('Inbound KB/interval')
ax_net2.set_ylabel('Outbound KB/interval')
ax_net.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.setp(ax_net.xaxis.get_majorticklabels(), rotation=30, ha='right')

plot_metric(axes[2, 1], 'memory_used_mb', 'MB', 'Memory Used')

plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'metrics_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: metrics_timeseries.png')

## 4. Statistical Summary

In [ ]:
mean_cols = [f'{c}_mean' for c in METRIC_COLS]
summary = df[mean_cols].describe().round(3)
summary.columns = [c.replace('_mean', '') for c in summary.columns]
summary

In [ ]:
# Correlation heatmap
corr = df[mean_cols].corr()
corr.columns = [c.replace('_mean', '') for c in corr.columns]
corr.index = corr.columns

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    ax=ax,
)
ax.set_title('Metric Correlation Heatmap')
plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'metrics_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: metrics_correlation.png')

## 5. Export to Parquet

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
parquet_path = EXPORTS_DIR / f'metrics_{timestamp}.parquet'

table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, str(parquet_path), compression='snappy')

size_kb = parquet_path.stat().st_size / 1024
print(f'Exported: {parquet_path.name} ({size_kb:.1f} KB, {len(df)} rows)')
print(f'Download via: GET http://<backend-ip>:8000/exports/{parquet_path.name}')

In [ ]:
# Optional: upload to GCS
GCS_BUCKET = os.environ.get('GCS_BUCKET_NAME')

if GCS_BUCKET:
    from google.cloud import storage as gcs
    gcs_client = gcs.Client()
    bucket = gcs_client.bucket(GCS_BUCKET)
    blob = bucket.blob(f'notebook-exports/{parquet_path.name}')
    blob.upload_from_filename(str(parquet_path))
    print(f'Uploaded to gs://{GCS_BUCKET}/notebook-exports/{parquet_path.name}')
else:
    print('GCS_BUCKET_NAME not set — skipping GCS upload')

In [ ]:
# Verify by reading back the Parquet file
df_verify = pd.read_parquet(str(parquet_path))
print(f'Verification read: {len(df_verify)} rows, {df_verify.shape[1]} columns')
df_verify.dtypes

In [ ]:
# Close MongoDB connection
client.close()
print('Done. MongoDB connection closed.')